In [0]:

!pip install openai kneed hdbscan umap-learn 
dbutils.library.restartPython()


In [0]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import os
import glob
from collections import defaultdict, Counter
from datetime import datetime
import warnings
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed
import logging
# from tqdm import tqdm

warnings.filterwarnings('ignore')

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class AgentAnalyzer:
    def __init__(self, json_folder_path, output_folder="analysis_results"):
        """
        Initialize the bulk analyzer for processing thousands of JSON files
        
        Args:
            json_folder_path (str): Path to folder containing JSON files
            output_folder (str): Path to output folder for results
        """
        self.json_folder_path = Path(json_folder_path)
        self.output_folder = Path(output_folder)
        self.output_folder.mkdir(exist_ok=True)
        
        self.all_interactions = []
        self.agent_metrics = None
        self.file_processing_stats = {}
        
    def discover_json_files(self):
        """Discover all JSON files in the specified folder"""
        json_files = list(self.json_folder_path.glob("*.json"))
        logger.info(f"Discovered {len(json_files)} JSON files")
        return json_files
    
    def process_single_file(self, file_path):
        """Process a single JSON file and extract agent interaction data"""
        try:
            with open(file_path, 'r', encoding='utf-8') as file:
                data = json.load(file)
            
            file_interactions = []
            
            # Handle both single interaction and list of interactions
            if isinstance(data, list):
                interactions_list = data
            else:
                interactions_list = [data]
            
            for interaction in interactions_list:
                # Extract basic interaction metadata
                interaction_metadata = interaction.get('Conversational analysis', {}).get('interaction_metadata', {})
                ccts_relation = interaction.get('Conversational analysis', {}).get('ccts_relation', {})
                interaction_summary = interaction.get('Conversational analysis', {}).get('interaction_summary', {})
                escalation_factors = interaction.get('Conversational analysis', {}).get('escalation_factors', {})
                if interaction_summary.get('identified_main_issue'):
                    main_issue = interaction_summary.get('identified_main_issue')
                else:
                    main_issue = interaction_summary.get('identified main issue')
                base_interaction_data = {
                    'file_name': file_path.name,
                    'interaction_index': interaction.get('interaction_index'),
                    'calendar_date': interaction.get('calendar_date'),
                    'case_number': interaction.get('case_number'),
                    'media_type': interaction_metadata.get('media_type'),
                    'talk_time_seconds': interaction_metadata.get('talk_time_seconds', 0),
                    'customer_sentiment': interaction_summary.get('customer_sentiment',''),
                    'interaction_summary': interaction_summary,
                    'structured_summary': interaction_summary.get('structured_summary',''),
                    "identified main issue": main_issue,
                    # 'ccts_relation': ccts_relation,
                    'resolution_status': interaction_summary.get('resolution_status'),
                    'customer_intent': interaction_summary.get('customer_intent'),
                    'escalation_risk_score': escalation_factors.get('escalation_risk_score'),
                    'customer_callbacks': escalation_factors.get('customer_callbacks'),
                    'internal_transfer': escalation_factors.get('interal_transfer'),  
                    'dropped_calls': escalation_factors.get('dropped_calls'),
                    'notable_moments': interaction_metadata.get('notable_moments', []),
                    'escakation_contributing_factors': escalation_factors.get('contributing_factors', []),
                    'customer_frustration_points': escalation_factors.get('customer_frustration_points', [])
                }
                
                # Extract agent-specific evaluations
                agent_evaluations = interaction.get('Agent Evaluation', {}).get('agent_evaluations', {})
        
                for agent_eval in agent_evaluations:
                    agent_evaluation = agent_eval.get('agent_evaluation', {})
                    infraction_assessment = agent_eval.get('infraction_assessment', {})
                    internal_escalation = agent_eval.get('internal_escalation', {})
                        
                    agent_record = base_interaction_data.copy()
                    agent_record.update({
                        'agent_identifier': agent_eval.get('agent_identifier'),
                        'overall_performance': agent_evaluation.get('overall_performance'),
                        'communication_skills': agent_evaluation.get('communication_skills'),
                        'empathy_level': agent_evaluation.get('empathy_level'),
                        'professionalism_level': agent_evaluation.get('professionalism_level'),
                        'problem_resolution_level': agent_evaluation.get('problem_resolution_level'),
                        'resolution_efficiency': agent_evaluation.get('resolution_efficiency'),
                        'key_strengths': agent_evaluation.get('key_strengths', []),
                        'improvement_areas': agent_evaluation.get('improvement_areas', []),
                        'key_strengths': agent_evaluation.get('key_strengths', []),
                        'improvement_areas': agent_evaluation.get('improvement_areas', []),
                        'agent_infraction': infraction_assessment.get('agent_infraction'),
                        'educational_gap_detected': infraction_assessment.get('educational_gap_detected'),
                        'educational_gap_details': infraction_assessment.get('educational_gap_details'),
                        'unactioned_threat_detection': infraction_assessment.get('unactioned_threat_detection'),
                        'infraction_severity_on_ccts': infraction_assessment.get('infraction_severity_on_ccts'),
                        'coachback_opportunities': infraction_assessment.get('coachback_opportunities'),
                        'reached_manager': internal_escalation.get('reached_manager'),
                        'manager_escalation_appropriate': internal_escalation.get('manager_escalation_appropriate'),
                        'escalation_timing': internal_escalation.get('escalation_timing'),
                        'missed_escalation_opportunity': internal_escalation.get('missed_escalation_opportunity'),
                        'escalation_impact': internal_escalation.get('escalation_impact'),
                        'overall evaluation': agent_evaluation
                        # 'overall_evaluation_reason': agent_evaluation.get('overall_evaluation_reason')'
                    })
                        
                    file_interactions.append(agent_record)

            
            return {
                'file_path': str(file_path),
                'interactions': file_interactions,
                'interaction_count': len(file_interactions),
                'status': 'success'
            }
            
        except Exception as e:
            logger.error(f"Error processing file {file_path}: {str(e)}")
            return {
                'file_path': str(file_path),
                'interactions': [],
                'interaction_count': 0,
                'status': 'error',
                'error': str(e)
            }
    
    def process_files_parallel(self, max_workers=None):
        """Process all JSON files in parallel"""
        json_files = self.discover_json_files()
        
        if not json_files:
            logger.warning("No JSON files found in the specified directory")
            return
        
        if max_workers is None:
            max_workers = min(mp.cpu_count(), len(json_files))
        
        logger.info(f"Processing {len(json_files)} files using {max_workers} workers")
        
        all_interactions = []
        processing_stats = {
            'total_files': len(json_files),
            'successful_files': 0,
            'failed_files': 0,
            'total_interactions': 0
        }
        
        with ProcessPoolExecutor(max_workers=max_workers) as executor:
            # Submit all files for processing
            future_to_file = {executor.submit(self.process_single_file, file_path): file_path 
                             for file_path in json_files}
            
            # Process completed futures with progress bar
            for future in as_completed(future_to_file):
                result = future.result()
                
                if result['status'] == 'success':
                    all_interactions.extend(result['interactions'])
                    processing_stats['successful_files'] += 1
                    processing_stats['total_interactions'] += result['interaction_count']
                else:
                    processing_stats['failed_files'] += 1
                
                self.file_processing_stats[result['file_path']] = result
        
        self.all_interactions = all_interactions
        logger.info(f"Processing complete: {processing_stats['successful_files']} successful, "
                   f"{processing_stats['failed_files']} failed, "
                   f"{processing_stats['total_interactions']} total interactions")
        
        return processing_stats
    
    def create_performance_scores(self):
        """Convert qualitative assessments to numerical scores"""
        if not self.all_interactions:
            logger.error("No interactions data available. Run process_files_parallel() first.")
            return None
        
        df = pd.DataFrame(self.all_interactions)
        
        # Define scoring mappings
        skill_scores = {
            'Excellent': 4, 'Good': 3,  'Fair': 2, 'Poor': 1,
            'High': 3, 'Medium': 2, 'Low': 1, 'Average': 3
        }
        
        sentiment_scores = {
            'satisfied': 5, 'neutral': 3, 'frustrated': 1, 'angry': 0
        }
        
        resolution_scores = {
            'resolved': 5, 'partially_resolved': 3, 'unresolved': 1,
            'Inefficient': 1, 'Average': 2,  'Efficient' :3, 'Highly Efficient':4 
        }
        
        risk_scores = {
            'low': 1, 'medium': 2, 'high': 3
        }
        professionalism_level ={
            'Poor': 1, 'Fair': 2 , 'Good' :3, 'Excellent':4
        }

        
        # Apply scoring with error handling
        df['communication_score'] = df['communication_skills'].map(skill_scores).fillna(3)
        df['empathy_score'] = df['empathy_level'].map(skill_scores).fillna(3)
        df['professionalism_score'] = df['professionalism_level'].map(professionalism_level).fillna(3)
        df['resolution_score'] = df['problem_resolution_level'].map(professionalism_level).fillna(3)
        df['efficiency_score'] = df['resolution_efficiency'].map(resolution_scores).fillna(3)
        df['sentiment_score'] = df['customer_sentiment'].map(sentiment_scores).fillna(3)
        df['resolution_status_score'] = df['resolution_status'].map(resolution_scores).fillna(3)
        df['risk_score'] = df['escalation_risk_score'].map(risk_scores).fillna(2)
        
        # Calculate composite performance score
        df['composite_performance_score'] = (
            df['communication_score'] * 0.25 +
            df['empathy_score'] * 0.1 +
            df['professionalism_score'] * 0.15 +
            df['resolution_score'] * 0.25 +
            df['efficiency_score'] * 0.25
        )
        
        # Convert boolean-like strings to actual booleans for analysis
        boolean_columns = ['agent_infraction', 'educational_gap_detected', 'customer_callbacks', 
                          'internal_transfer', 'dropped_calls', 'reached_manager', 
                          'manager_escalation_appropriate', 'missed_escalation_opportunity']
        
        for col in boolean_columns:
            if col in df.columns:
                df[f'{col}_bool'] = df[col].map({'Yes': True, 'No': False}).fillna(False)
        
        self.agent_metrics = df
        return df
    
    def generate_agent_summary_statistics(self):
        """Generate comprehensive agent performance statistics"""
        if self.agent_metrics is None:
            logger.error("Agent metrics not available. Run create_performance_scores() first.")
            return None
        
        df = self.agent_metrics
        
        # Group by agent and calculate comprehensive statistics
        agent_stats = df.groupby('agent_identifier').agg({
            # Performance scores
            'composite_performance_score': ['mean', 'std', 'min', 'max', 'count'],
            'communication_score': ['mean', 'std'],
            'empathy_score': ['mean', 'std'],
            'professionalism_score': ['mean', 'std'],
            'resolution_score': ['mean', 'std'],
            'efficiency_score': ['mean', 'std'],
            
            # Customer metrics
            'sentiment_score': ['mean', 'std'],
            'resolution_status_score': ['mean', 'std'],
            'risk_score': ['mean', 'std'],
            
            # Interaction metrics
            'talk_time_seconds': ['mean', 'std', 'median'],
            'notable_moments_count': ['mean', 'sum'],
            'contributing_factors_count': ['mean', 'sum'],
            'customer_frustration_points_count': ['mean', 'sum'],
            
            # Performance indicators
            'key_strengths_count': ['mean', 'sum'],
            'improvement_areas_count': ['mean', 'sum'],
            
            # Boolean metrics (using the _bool columns)
            'agent_infraction_bool': ['sum', 'mean'],
            'educational_gap_detected_bool': ['sum', 'mean'],
            'customer_callbacks_bool': ['sum', 'mean'],
            'internal_transfer_bool': ['sum', 'mean'],
            'dropped_calls_bool': ['sum', 'mean'],
            'reached_manager_bool': ['sum', 'mean'],
            'missed_escalation_opportunity_bool': ['sum', 'mean'],
            
            # Date range
            'calendar_date': ['min', 'max', 'nunique']
        }).round(3)
        
        # Flatten column names
        agent_stats.columns = ['_'.join(col).strip() for col in agent_stats.columns]
        agent_stats = agent_stats.reset_index()
        
        # Calculate additional derived metrics
        agent_stats['performance_consistency'] = (
            agent_stats['composite_performance_score_std'] / 
            agent_stats['composite_performance_score_mean']
        ).fillna(0)
        
        agent_stats['infraction_rate'] = agent_stats['agent_infraction_bool_mean']
        agent_stats['escalation_miss_rate'] = agent_stats['missed_escalation_opportunity_bool_mean']
        agent_stats['customer_callback_rate'] = agent_stats['customer_callbacks_bool_mean']
        
        return agent_stats

In [0]:
path = "/Workspace/Users/morgan.wang@rci.rogers.ca/Conversation_2501/"
analyst = AgentAnalyzer(path)
analyst.process_files_parallel()
interactions = analyst.all_interactions
df = analyst.create_performance_scores()

In [0]:
df.head(5)

In [0]:
 df.to_pickle("converstations_full_df.pkl")

In [0]:
file = "/Workspace/Users/morgan.wang@rci.rogers.ca/Coversation_full_dataframe.pkl"
with open(file, "rb") as f:
    df = pd.read_pickle(f)

In [0]:
df.head(5)

In [0]:
performances = df['overall evaluation'].astype(str)

In [0]:
from openai import AzureOpenAI
import openai
import time
# from openai.error import RateLimitError


client = AzureOpenAI(
    api_key='28f2878eb3244d7083072353c59d2bb7',  
    api_version="2024-02-01",
    azure_endpoint="https://mazcaeprdmkencicognitiveopenai01.openai.azure.com/"
)

def create_embeddings_with_sleep(client, model, input_data, max_retries=3, wait_time=61):
    retries = 0
    while retries < max_retries:
        try:
            performance_embeds = client.embeddings.create(
                model=model,
                input=input_data
            )
            return performance_embeds
        except openai.RateLimitError as e:  # Corrected this line
            retries += 1
            if retries < max_retries:
                # Exponential backoff strategy
                wait_time *= 2
                time.sleep(wait_time)
            else:
                raise
# embeddings = create_embeddings_with_sleep(client= client, model='text-embedding-ada-002', input_data=performances.to_list())

In [0]:
from openai import OpenAI
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
print(DATABRICKS_TOKEN)
client2 = OpenAI(
    api_key=DATABRICKS_TOKEN,
    base_url="https://adb-5880834975464433.13.azuredatabricks.net/serving-endpoints"
)
# agent_performance_list = performances.to_list()
# input_data = [{"text": performance} for performance in agent_performance_list]



In [0]:
embeds = create_embeddings_with_sleep(client= client, model='text-embedding-ada-002', input_data=performances.to_list())

In [0]:
chunk_size = 1000

# Convert performances to a list
performances_list = performances.to_list()

# Split the performances data into smaller chunks
chunks = [performances_list[i:i + chunk_size] for i in range(0, len(performances_list), chunk_size)]

# Initialize an empty list to store all embeddings
all_embeds = []

# Create embeddings for each chunk and aggregate them
for chunk in chunks:
    embeds = client2.embeddings.create(
        model="embeddings",
        input=chunk
    )
    all_embeds.extend(embeds.data)

# Convert the aggregated embeddings to a numpy array
embed_array = np.array([embed.embedding for embed in all_embeds])

optimal k and dimension reduction


In [0]:
import pandas as pd
import numpy as np
import json
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics.pairwise import cosine_similarity, cosine_distances
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from kneed import KneeLocator
from sklearn.metrics import silhouette_score
from datetime import datetime, timedelta
import hdbscan
import warnings
warnings.filterwarnings("ignore")
import time
import matplotlib.pyplot as plt
 
try:
    import umap.umap_ as umap
 
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False
    print("UMAP not available, using PCA for dimension reduction")
    
def apply_dimension_reduction(
    embeddings_array, method="auto", target_dim=None, explained_variance_threshold=0.7
):
    """
    Apply dimension reduction to embeddings
 
    Args:
        embeddings_array: Input embeddings
        method: 'pca', 'umap', or 'auto'
        target_dim: Target dimensions (if None, will be determined automatically)
        explained_variance_threshold: For PCA, minimum explained variance to retain
 
    Returns:
        reduced_embeddings, reducer_object, reduction_info
    """
    print(f"Original embedding dimensions: {embeddings_array.shape}")
 
    # Standardize the embeddings
    # scaler = StandardScaler()
    # scaled_embeddings = scaler.fit_transform(embeddings_array)
 
    reduction_info = {
        "original_dim": embeddings_array.shape[1],
        "method": method,
        # "scaler": scaler,
    }
 
    if method == "auto":
        # Choose method based on data size and availability
        if len(embeddings_array) > 1000 and UMAP_AVAILABLE:
            method = "umap"
        else:
            method = "pca"
        reduction_info["method"] = method
 
    if method == "pca":
        # Determine optimal number of components for PCA
        if target_dim is None:
            # First, fit PCA with all components to analyze explained variance
            pca_full = PCA()
            pca_full.fit(embeddings_array)
 
            # Find number of components needed for desired explained variance
            cumsum_var = np.cumsum(pca_full.explained_variance_ratio_)
            target_dim = np.argmax(cumsum_var >= explained_variance_threshold) + 1
            target_dim = min(
                target_dim, embeddings_array.shape[1]
            )
            target_dim = max(target_dim, 5)  # Minimum 5 dimensions
            # target_dim = min(target_dim, 15)
        # Apply PCA with determined number of components
        pca = PCA(n_components=target_dim, random_state=42)
        reduced_embeddings = pca.fit_transform(embeddings_array)
 
        explained_variance = np.sum(pca.explained_variance_ratio_)
        reduction_info.update(
            {
                "reduced_dim": target_dim,
                "explained_variance": explained_variance,
                "explained_variance_ratio": pca.explained_variance_ratio_,
                "reducer": pca,
            }
        )
 
        print(
            f"PCA: Reduced to {target_dim} dimensions, explained variance: {explained_variance:.3f}"
        )
 
    elif method == "umap" and UMAP_AVAILABLE:
        # Determine optimal number of components for UMAP
        if target_dim is None:
            # Use heuristic: sqrt of original dimensions, but at least 5 and at most 30
            target_dim = max(5, min(30, int(np.sqrt(embeddings_array.shape[1]))))
 
        # UMAP parameters
        n_neighbors = min(30, len(embeddings_array) - 1)
        min_dist = 0.3
 
        umap_reducer = umap.UMAP(
            n_components=target_dim,
            n_neighbors=n_neighbors,
            min_dist=min_dist,
            random_state=42,
            metric="cosine",
        )
 
        reduced_embeddings = umap_reducer.fit_transform(embeddings_array)
 
        reduction_info.update(
            {
                "reduced_dim": target_dim,
                "n_neighbors": n_neighbors,
                "min_dist": min_dist,
                "reducer": umap_reducer,
            }
        )
 
        print(f"UMAP: Reduced to {target_dim} dimensions")
 
    else:
        # Fallback to PCA if UMAP not available
        print("Falling back to PCA...")
        return apply_dimension_reduction(
            embeddings_array,
            method="pca",
            target_dim=target_dim,
            explained_variance_threshold=explained_variance_threshold,
        )
 
    return reduced_embeddings, reduction_info
 
 
def determine_optimal_k(embeddings_array, min_clusters=2, max_clusters=8):
    """
    Determine the optimal number of clusters using both the elbow method and silhouette scores
    """
    # Ensure we don't try more clusters than we have samples
    max_possible_clusters = min(max_clusters, len(embeddings_array) - 1)
    if max_possible_clusters <= min_clusters:
        return min_clusters
 
    # Calculate inertia (for elbow method) and silhouette scores
    inertia_values = []
    silhouette_values = []
    k_values = range(min_clusters, max_possible_clusters + 1)
 
    for k in k_values:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans.fit(embeddings_array)
        inertia_values.append(kmeans.inertia_)
 
        # Only calculate silhouette if we have enough samples and more than one cluster
        if len(embeddings_array) > k and k > 1:
            silhouette_avg = silhouette_score(embeddings_array, kmeans.labels_)
            silhouette_values.append(silhouette_avg)
        else:
            silhouette_values.append(0)
 
    # Try to find the elbow point using the KneeLocator
    try:
        kneedle = KneeLocator(
            list(k_values),
            inertia_values,
            S=1.0,
            curve="convex",
            direction="decreasing",
        )
        elbow_k = kneedle.elbow
    except Exception:
        elbow_k = None
 
    # Find the k with the highest silhouette score
    best_silhouette_k = (
        k_values[np.argmax(silhouette_values)] if silhouette_values else None
    )
 
    # Plot the results
    plt.figure(figsize=(12, 5))
 
    # Plot inertia
    plt.subplot(1, 2, 1)
    plt.plot(k_values, inertia_values, "bo-")
    if elbow_k:
        plt.axvline(x=elbow_k, color="r", linestyle="--")
        plt.text(elbow_k + 0.1, np.mean(inertia_values), f"Elbow at k={elbow_k}")
    plt.xlabel("Number of clusters (k)")
    plt.ylabel("Inertia")
    plt.title("Elbow Method")
 
    # Plot silhouette
    plt.subplot(1, 2, 2)
    plt.plot(k_values, silhouette_values, "go-")
    if best_silhouette_k:
        plt.axvline(x=best_silhouette_k, color="r", linestyle="--")
        plt.text(
            best_silhouette_k + 0.1,
            np.mean(silhouette_values),
            f"Best at k={best_silhouette_k}",
        )
    plt.xlabel("Number of clusters (k)")
    plt.ylabel("Silhouette Score")
    plt.title("Silhouette Method")
 
    plt.tight_layout()
    plt.savefig(f"optimal_k_analysis_rc.png")
    plt.close()
 
    # Decision logic for optimal k
    if elbow_k and best_silhouette_k:
        # If both methods agree, use that value
        if elbow_k == best_silhouette_k:
            optimal_k = elbow_k
        # Otherwise, prefer silhouette if it's reasonable
        elif best_silhouette_k >= min_clusters and best_silhouette_k <= max_clusters:
            optimal_k = best_silhouette_k
        else:
            optimal_k = elbow_k
    elif elbow_k:
        optimal_k = elbow_k
    elif best_silhouette_k:
        optimal_k = best_silhouette_k
    else:
        # Default to middle of range if methods fail
        optimal_k = (min_clusters + max_possible_clusters) // 2
 
    # Ensure the optimal k is within our desired range
    optimal_k = max(min_clusters, min(optimal_k, max_possible_clusters))
 
    return optimal_k

In [0]:
embed_array = np.array([embed.embedding for embed in embeds.data])

In [0]:
determine_optimal_k(embed_array, min_clusters=2, max_clusters=8)

In [0]:
import plotly.express as px
import plotly.graph_objects as go
performance_issue = performances_list
# embedding_vectors = np.array([emb.embedding for emb in embeds.data])
embedding_vectors = embed_array
embedding_vectors,  _  = apply_dimension_reduction(embedding_vectors, method="auto")
n = determine_optimal_k(embed_array, min_clusters=2, max_clusters=8)
pca = PCA(n_components=2, random_state=42)
embeddings_2d_pca = pca.fit_transform(embedding_vectors)
kmeans = KMeans(n_clusters=k, random_state=42)
cluster_labels = kmeans.fit_predict(embedding_vectors)
embedding_lists = [emb.tolist() for emb in embedding_vectors]
cluster_df = pd.DataFrame(
            {"agent summary": performance_issue, "Cluster": cluster_labels, "embedding": embedding_lists}
        )
reducer = umap.UMAP(n_components=2, random_state=42)
embeddings_2d_umap = reducer.fit_transform(embedding_vectors) 
# print(f"✅ Created {len(set(cluster_labels))} clusters")

# Method 2: t-SNE (better for visualization but slower)
tsne = TSNE(n_components=2, random_state=42)
embeddings_2d_tsne = tsne.fit_transform(embedding_vectors)

# Create visualization dataframes
df_pca = pd.DataFrame({
    'x': embeddings_2d_pca[:, 0],
    'y': embeddings_2d_pca[:, 1],
    'cluster': cluster_labels,
    'text':  performance_issue
})

df_tsne = pd.DataFrame({
    'x': embeddings_2d_tsne[:, 0],
    'y': embeddings_2d_tsne[:, 1],
    'cluster': cluster_labels,
    'text':  performance_issue
})
df_umap = pd.DataFrame({
    'x': embeddings_2d_umap[:, 0],
    'y': embeddings_2d_umap[:, 1],
    'cluster': cluster_labels,
    'text':  performance_issue
})
# Visualization 1: PCA with matplotlib
plt.figure(figsize=(12, 8))
scatter = plt.scatter(embeddings_2d_pca[:, 0], embeddings_2d_pca[:, 1], 
                     c=cluster_labels, cmap='tab10', alpha=0.7)
plt.colorbar(scatter)
plt.title(f'K-means Clustering of Agent Performance (PCA) - {n} Clusters')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)')

# Add cluster centers
centers_2d = pca.transform(kmeans.cluster_centers_)
plt.scatter(centers_2d[:, 0], centers_2d[:, 1], 
           c='red', marker='x', s=200, linewidths=3, label='Centroids')
plt.legend()
plt.show()

# Visualization 2: t-SNE with matplotlib
plt.figure(figsize=(12, 8))
scatter = plt.scatter(embeddings_2d_tsne[:, 0], embeddings_2d_tsne[:, 1], 
                     c=cluster_labels, cmap='tab10', alpha=0.7)
plt.colorbar(scatter)
plt.title(f'K-means Clustering of Agent Performance (t-SNE) - {n} Clusters')
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')



# Visualization 3: UMAP with matplotlib
plt.figure(figsize=(12, 8))
scatter = plt.scatter(embeddings_2d_umap[:, 0], embeddings_2d_umap[:, 1], 
                     c=cluster_labels, cmap='tab10', alpha=0.7)
plt.colorbar(scatter)
plt.title(f'K-means Clustering of Agent Performance (t-SNE) - {n} Clusters')
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.show()

# Interactive visualization with Plotly
fig_pca = px.scatter(df_pca, x='x', y='y', color='cluster', 
                     hover_data=['text'], 
                     title=f'Interactive K-means Clustering (PCA) - {n} Clusters',
                     labels={'x': f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)',
                            'y': f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)'})
fig_pca.show()

fig_tsne = px.scatter(df_tsne, x='x', y='y', color='cluster', 
                      hover_data=['text'], 
                      title=f'Interactive K-means Clustering (t-SNE) - {n} Clusters',
                      labels={'x': 't-SNE 1', 'y': 't-SNE 2'})
fig_tsne.show()

# Analyze clusters
print("\nCluster Analysis:")
print("="*50)
for i in range(n):
    cluster_data = cluster_df[cluster_df['Cluster'] == i]
    print(f"\nCluster {i} ({len(cluster_data)} items):")
    print("-" * 30)
    
    # Show sample issues from this cluster
    sample_issues = cluster_data["agent summary"].head(3).tolist()
    for j, issue in enumerate(sample_issues, 1):
        print(f"{j}. {issue[:100]}...")
    
    if len(cluster_data) > 3:
        print(f"... and {len(cluster_data) - 3} more")

# Cluster statistics
cluster_stats = cluster_df.groupby('Cluster').size().reset_index(name='Count')
cluster_stats['Percentage'] = (cluster_stats['Count'] / len(performance_issue) * 100).round(2)

print(f"\nCluster Distribution:")
print(cluster_stats)

# Visualize cluster distribution
plt.figure(figsize=(10, 6))
plt.bar(cluster_stats['Cluster'], cluster_stats['Count'])
plt.xlabel('Cluster')
plt.ylabel('Number of Issues')
plt.title('Distribution of Customer Issues Across Clusters')
for i, v in enumerate(cluster_stats['Count']):
    plt.text(i, v + 0.5, str(v), ha='center')
plt.show()

In [0]:
cluster_df[cluster_df['Cluster'] == 2 ].iloc[0]['agent summary']

In [0]:
performance_issue = (
    ' Agent overall_performance: ' + df['overall_performance'].astype(str) + 
    ' Customer key issue: ' + df['identified main issue'].astype(str)
)
performance_issue
# embeddings = create_embeddings_with_sleep(client=client, model='text-embedding-ada-002', input_data=performance_issue.to_list())

In [0]:
performance = df['overall evaluation'].astype(str)

In [0]:
performance_embeds = client2.embeddings.create(
    model="embeddings",
    input=performance.to_list()
)

In [0]:
df.head(3)

In [0]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.feature_extraction.text import TfidfVectorizer
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns
# from sentence_transformers import SentenceTransformer
import umap

class ImprovedClusteringAnalyzer:
    def __init__(self, performance_data):
        self.performance_data = performance_data
        self.embedding_vectors = None
        self.cluster_labels = None
        self.optimal_k = None
        
    def preprocess_embeddings(self, embedding_vectors):
        """Improve embedding preprocessing"""
        # 1. Normalize embeddings (very important for K-means)
        from sklearn.preprocessing import normalize
        normalized_embeddings = normalize(embedding_vectors, norm='l2')
        
        # 2. Optional: Apply PCA for dimensionality reduction while preserving variance
        if embedding_vectors.shape[1] > 100:  # Only if high dimensional
            pca = PCA(n_components=min(100, embedding_vectors.shape[0]//2), random_state=42)
            reduced_embeddings = pca.fit_transform(normalized_embeddings)
            print(f"Reduced dimensions from {embedding_vectors.shape[1]} to {reduced_embeddings.shape[1]}")
            print(f"Explained variance: {pca.explained_variance_ratio_.sum():.3f}")
            return reduced_embeddings, pca
        
        return normalized_embeddings, None
    
    def find_optimal_k_improved(self, embeddings, max_k=15):
        """Improved method to find optimal K using multiple metrics"""
        if len(embeddings) < 4:
            return 2
            
        max_k = min(max_k, len(embeddings) - 1)
        
        metrics = {
            'inertia': [],
            'silhouette': [],
            'calinski_harabasz': [],
            'davies_bouldin': []
        }
        
        k_range = range(2, max_k + 1)
        
        for k in k_range:
            # Use multiple random initializations
            kmeans = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
            labels = kmeans.fit_predict(embeddings)
            
            metrics['inertia'].append(kmeans.inertia_)
            metrics['silhouette'].append(silhouette_score(embeddings, labels))
            metrics['calinski_harabasz'].append(calinski_harabasz_score(embeddings, labels))
            metrics['davies_bouldin'].append(davies_bouldin_score(embeddings, labels))
        
        # Find optimal K using elbow method for inertia
        def find_elbow(values):
            n_points = len(values)
            all_coord = np.vstack((range(n_points), values)).T
            first_point = all_coord[0]
            line_vec = all_coord[-1] - all_coord[0]
            line_vec_norm = line_vec / np.sqrt(np.sum(line_vec**2))
            
            vec_from_first = all_coord - first_point
            scalar_product = np.sum(vec_from_first * line_vec_norm, axis=1)
            vec_from_first_parallel = np.outer(scalar_product, line_vec_norm)
            vec_to_line = vec_from_first - vec_from_first_parallel
            dist_to_line = np.sqrt(np.sum(vec_to_line ** 2, axis=1))
            
            return np.argmax(dist_to_line)
        
        elbow_k = find_elbow(metrics['inertia']) + 2
        silhouette_k = np.argmax(metrics['silhouette']) + 2
        
        # Combine metrics (weighted average)
        optimal_k = int(np.round((elbow_k * 0.4 + silhouette_k * 0.6)))
        
        print(f"Elbow method suggests K={elbow_k}")
        print(f"Silhouette method suggests K={silhouette_k}")
        print(f"Combined optimal K={optimal_k}")
        
        # Plot metrics
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        axes[0,0].plot(k_range, metrics['inertia'], 'bo-')
        axes[0,0].axvline(x=elbow_k, color='r', linestyle='--', label=f'Elbow K={elbow_k}')
        axes[0,0].set_title('Elbow Method')
        axes[0,0].set_xlabel('K')
        axes[0,0].set_ylabel('Inertia')
        axes[0,0].legend()
        
        axes[0,1].plot(k_range, metrics['silhouette'], 'go-')
        axes[0,1].axvline(x=silhouette_k, color='r', linestyle='--', label=f'Best K={silhouette_k}')
        axes[0,1].set_title('Silhouette Score')
        axes[0,1].set_xlabel('K')
        axes[0,1].set_ylabel('Silhouette Score')
        axes[0,1].legend()
        
        axes[1,0].plot(k_range, metrics['calinski_harabasz'], 'mo-')
        axes[1,0].set_title('Calinski-Harabasz Score')
        axes[1,0].set_xlabel('K')
        axes[1,0].set_ylabel('CH Score')
        
        axes[1,1].plot(k_range, metrics['davies_bouldin'], 'co-')
        axes[1,1].set_title('Davies-Bouldin Score')
        axes[1,1].set_xlabel('K')
        axes[1,1].set_ylabel('DB Score')
        
        plt.tight_layout()
        plt.show()
        
        return optimal_k, metrics
    
    def compare_clustering_algorithms(self, embeddings):
        """Compare different clustering algorithms"""
        algorithms = {}
        
        # K-means with optimal K
        optimal_k, _ = self.find_optimal_k_improved(embeddings)
        kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
        algorithms['K-Means'] = kmeans.fit_predict(embeddings)
        
        # DBSCAN (density-based)
        from sklearn.neighbors import NearestNeighbors
        neighbors = NearestNeighbors(n_neighbors=4)
        neighbors_fit = neighbors.fit(embeddings)
        distances, indices = neighbors_fit.kneighbors(embeddings)
        distances = np.sort(distances, axis=0)
        distances = distances[:,1]
        eps = np.percentile(distances, 90)  # Use 90th percentile as eps
        
        dbscan = DBSCAN(eps=eps, min_samples=max(2, len(embeddings)//20))
        algorithms['DBSCAN'] = dbscan.fit_predict(embeddings)
        
        # Agglomerative Clustering
        agg = AgglomerativeClustering(n_clusters=optimal_k)
        algorithms['Agglomerative'] = agg.fit_predict(embeddings)
        
        # Evaluate each algorithm
        results = {}
        for name, labels in algorithms.items():
            if len(set(labels)) > 1 and -1 not in labels:  # Valid clustering
                silhouette = silhouette_score(embeddings, labels)
                results[name] = {
                    'labels': labels,
                    'silhouette': silhouette,
                    'n_clusters': len(set(labels)),
                    'n_noise': sum(labels == -1) if -1 in labels else 0
                }
            elif name == 'DBSCAN' and -1 in labels:  # DBSCAN with noise
                valid_labels = labels[labels != -1]
                valid_embeddings = embeddings[labels != -1]
                if len(set(valid_labels)) > 1:
                    silhouette = silhouette_score(valid_embeddings, valid_labels)
                    results[name] = {
                        'labels': labels,
                        'silhouette': silhouette,
                        'n_clusters': len(set(valid_labels)),
                        'n_noise': sum(labels == -1)
                    }
        
        # Print comparison
        print("\nClustering Algorithm Comparison:")
        print("="*50)
        for name, metrics in results.items():
            print(f"{name}:")
            print(f"  Silhouette Score: {metrics['silhouette']:.3f}")
            print(f"  Number of Clusters: {metrics['n_clusters']}")
            if metrics['n_noise'] > 0:
                print(f"  Noise Points: {metrics['n_noise']}")
            print()
        
        # Return best algorithm
        best_algo = max(results.keys(), key=lambda x: results[x]['silhouette'])
        print(f"Best algorithm: {best_algo}")
        
        return results, best_algo
    
    def advanced_visualization(self, embeddings, labels, performance_data):
        """Create advanced visualizations"""
        
        # 1. UMAP (often better than t-SNE)
        reducer = umap.UMAP(n_components=2, random_state=42)
        embeddings_2d_umap = reducer.fit_transform(embeddings)
        
        # 2. PCA
        pca = PCA(n_components=2, random_state=42)
        embeddings_2d_pca = pca.fit_transform(embeddings)
        
        # 3. t-SNE
        tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(embeddings)-1))
        embeddings_2d_tsne = tsne.fit_transform(embeddings)
        
        # Create comparison plot
        fig, axes = plt.subplots(1, 3, figsize=(20, 6))
        
        methods = [
            ('PCA', embeddings_2d_pca, f'PC1 ({pca.explained_variance_ratio_[0]:.1%})', f'PC2 ({pca.explained_variance_ratio_[1]:.1%})'),
            ('t-SNE', embeddings_2d_tsne, 't-SNE 1', 't-SNE 2'),
            ('UMAP', embeddings_2d_umap, 'UMAP 1', 'UMAP 2')
        ]
        
        for i, (method, coords, xlabel, ylabel) in enumerate(methods):
            scatter = axes[i].scatter(coords[:, 0], coords[:, 1], c=labels, cmap='tab10', alpha=0.7)
            axes[i].set_title(f'{method} Visualization')
            axes[i].set_xlabel(xlabel)
            axes[i].set_ylabel(ylabel)
            plt.colorbar(scatter, ax=axes[i])
        
        plt.tight_layout()
        plt.show()
        
        # Interactive UMAP plot
        df_umap = pd.DataFrame({
            'x': embeddings_2d_umap[:, 0],
            'y': embeddings_2d_umap[:, 1],
            'cluster': labels,
            'text': performance_data.astype(str).tolist()
        })
        
        fig_umap = px.scatter(df_umap, x='x', y='y', color='cluster',
                             hover_data=['text'],
                             title=f'Interactive UMAP Clustering - {len(set(labels))} Clusters')
        fig_umap.show()
        
        return embeddings_2d_umap
    
    def analyze_clusters_detailed(self, embeddings, labels, performance_data):
        """Detailed cluster analysis"""
        
        cluster_df = pd.DataFrame({
            'text': performance_data,
            'cluster': labels,
            'embedding': [emb.tolist() for emb in embeddings]
        })
        
        print("\nDetailed Cluster Analysis:")
        print("="*60)
        
        cluster_stats = []
        
        for cluster_id in sorted(set(labels)):
            if cluster_id == -1:  # Skip noise points in DBSCAN
                continue
                
            cluster_data = cluster_df[cluster_df['cluster'] == cluster_id]
            cluster_size = len(cluster_data)
            
            print(f"\n🔍 Cluster {cluster_id} ({cluster_size} items, {cluster_size/len(performance_data)*100:.1f}%)")
            print("-" * 50)
            
            # Sample texts
            sample_texts = cluster_data['text'].head(5).tolist()
            for j, text in enumerate(sample_texts, 1):
                print(f"{j}. {str(text)[:150]}...")
            
            if cluster_size > 5:
                print(f"... and {cluster_size - 5} more items")
            
            # Cluster statistics
            cluster_embeddings = np.array(cluster_data['embedding'].tolist())
            centroid = np.mean(cluster_embeddings, axis=0)
            
            # Intra-cluster distances
            distances = [np.linalg.norm(emb - centroid) for emb in cluster_embeddings]
            
            stats = {
                'cluster_id': cluster_id,
                'size': cluster_size,
                'percentage': cluster_size/len(performance_data)*100,
                'avg_distance_to_centroid': np.mean(distances),
                'std_distance_to_centroid': np.std(distances)
            }
            cluster_stats.append(stats)
            
            print(f"Average distance to centroid: {stats['avg_distance_to_centroid']:.3f}")
            print(f"Std distance to centroid: {stats['std_distance_to_centroid']:.3f}")
        
        # Overall statistics
        stats_df = pd.DataFrame(cluster_stats)
        
        print(f"\n📊 Overall Clustering Statistics:")
        print(f"Number of clusters: {len(set(labels)) - (1 if -1 in labels else 0)}")
        print(f"Silhouette score: {silhouette_score(embeddings, labels):.3f}")
        print(f"Largest cluster: {stats_df['size'].max()} items ({stats_df['percentage'].max():.1f}%)")
        print(f"Smallest cluster: {stats_df['size'].min()} items ({stats_df['percentage'].min():.1f}%)")
        print(f"Average cluster size: {stats_df['size'].mean():.1f} items")
        
        return cluster_df, stats_df

# Usage example



In [0]:
analyst = ImprovedClusteringAnalyzer(performances)
processed_embeddings, pca_reducer = analyzer.preprocess_embeddings(embeds)

In [0]:
def run_improved_clustering(performance_issue, embedding_vectors):
    """Run the improved clustering analysis"""
    
    analyzer = ImprovedClusteringAnalyzer(performance_issue)
    
    print("🚀 Starting Improved Clustering Analysis")
    print("="*60)
    
    # Step 1: Preprocess embeddings
    print("\n1️⃣ Preprocessing embeddings...")
    processed_embeddings, pca_reducer = analyzer.preprocess_embeddings(embedding_vectors)
    
    # Step 2: Find optimal parameters
    print("\n2️⃣ Finding optimal clustering parameters...")
    optimal_k, metrics = analyzer.find_optimal_k_improved(processed_embeddings)
    
    # Step 3: Compare different algorithms
    print("\n3️⃣ Comparing clustering algorithms...")
    algorithm_results, best_algorithm = analyzer.compare_clustering_algorithms(processed_embeddings)
    
    # Step 4: Use best algorithm for final clustering
    print(f"\n4️⃣ Applying {best_algorithm} for final clustering...")
    final_labels = algorithm_results[best_algorithm]['labels']
    
    # Step 5: Advanced visualization
    print("\n5️⃣ Creating advanced visualizations...")
    umap_coords = analyzer.advanced_visualization(processed_embeddings, final_labels, performance_issue)
    
    # Step 6: Detailed analysis
    print("\n6️⃣ Performing detailed cluster analysis...")
    cluster_df, stats_df = analyzer.analyze_clusters_detailed(processed_embeddings, final_labels, performance_issue)
    
    return {
        'analyzer': analyzer,
        'processed_embeddings': processed_embeddings,
        'labels': final_labels,
        'cluster_df': cluster_df,
        'stats_df': stats_df,
        'umap_coords': umap_coords,
        'metrics': metrics,
        'algorithm_results': algorithm_results
    }

run_improved_clustering(performances, embed_array)

In [0]:
cluster_df

In [0]:
def run_improved_clustering(performance_issue, embedding_vectors):
    """Run the improved clustering analysis"""
    
    analyzer = ImprovedClusteringAnalyzer(performance_issue)
    
    print("🚀 Starting Improved Clustering Analysis")
    print("="*60)
    
    # Step 1: Preprocess embeddings
    print("\n1️⃣ Preprocessing embeddings...")
    processed_embeddings, pca_reducer = analyzer.preprocess_embeddings(embedding_vectors)
    
    # Step 2: Find optimal parameters
    print("\n2️⃣ Finding optimal clustering parameters...")
    optimal_k, metrics = analyzer.find_optimal_k_improved(processed_embeddings)
    
    # Step 3: Compare different algorithms
    print("\n3️⃣ Comparing clustering algorithms...")
    algorithm_results, best_algorithm = analyzer.compare_clustering_algorithms(processed_embeddings)
    
    # Step 4: Use best algorithm for final clustering
    print(f"\n4️⃣ Applying {best_algorithm} for final clustering...")
    final_labels = algorithm_results[best_algorithm]['labels']
    
    # Step 5: Advanced visualization
    print("\n5️⃣ Creating advanced visualizations...")
    umap_coords = analyzer.advanced_visualization(processed_embeddings, final_labels, performance_issue)
    
    # Step 6: Detailed analysis
    print("\n6️⃣ Performing detailed cluster analysis...")
    cluster_df, stats_df = analyzer.analyze_clusters_detailed(processed_embeddings, final_labels, performance_issue)
    
    return {
        'analyzer': analyzer,
        'processed_embeddings': processed_embeddings,
        'labels': final_labels,
        'cluster_df': cluster_df,
        'stats_df': stats_df,
        'umap_coords': umap_coords,
        'metrics': metrics,
        'algorithm_results': algorithm_results
    }

# Alternative: Text-based clustering if embeddings aren't working well
def text_based_clustering_alternative(performance_issue):
    """Alternative approach using TF-IDF + clustering"""
    
    print("\n🔄 Alternative: Text-based Clustering")
    print("="*50)
    
    # Convert to string if not already
    texts = [str(text) for text in performance_issue]
    
    # TF-IDF Vectorization with better parameters
    vectorizer = TfidfVectorizer(
        max_features=1000,
        stop_words='english',
        ngram_range=(1, 2),  # Include bigrams
        min_df=2,  # Ignore terms that appear in less than 2 documents
        max_df=0.8  # Ignore terms that appear in more than 80% of documents
    )
    
    tfidf_matrix = vectorizer.fit_transform(texts)
    
    print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
    
    # Apply clustering
    analyzer = ImprovedClusteringAnalyzer(performance_issue)
    
    # Find optimal K
    optimal_k, metrics = analyzer.find_optimal_k_improved(tfidf_matrix.toarray())
    
    # Compare algorithms
    algorithm_results, best_algorithm = analyzer.compare_clustering_algorithms(tfidf_matrix.toarray())
    
    final_labels = algorithm_results[best_algorithm]['labels']
    
    # Visualization (reduce dimensions first)
    pca = PCA(n_components=50, random_state=42)
    reduced_tfidf = pca.fit_transform(tfidf_matrix.toarray())
    
    umap_coords = analyzer.advanced_visualization(reduced_tfidf, final_labels, performance_issue)
    
    # Analysis
    cluster_df, stats_df = analyzer.analyze_clusters_detailed(reduced_tfidf, final_labels, performance_issue)
    
    # Feature analysis - what terms characterize each cluster
    feature_names = vectorizer.get_feature_names_out()
    
    print("\n📝 Top Terms per Cluster:")
    print("="*40)
    
    for cluster_id in sorted(set(final_labels)):
        if cluster_id == -1:
            continue
            
        cluster_mask = final_labels == cluster_id
        cluster_tfidf = tfidf_matrix[cluster_mask]
        
        # Get mean TF-IDF scores for this cluster
        mean_scores = np.array(cluster_tfidf.mean(axis=0)).flatten()
        
        # Get top terms
        top_indices = mean_scores.argsort()[-10:][::-1]
        top_terms = [feature_names[i] for i in top_indices]
        top_scores = [mean_scores[i] for i in top_indices]
        
        print(f"\nCluster {cluster_id}:")
        for term, score in zip(top_terms, top_scores):
            print(f"  {term}: {score:.3f}")
    
    return {
        'vectorizer': vectorizer,
        'tfidf_matrix': tfidf_matrix,
        'labels': final_labels,
        'cluster_df': cluster_df,
        'stats_df': stats_df,
        'feature_names': feature_names
    }

# Ensemble clustering approach
def ensemble_clustering(performance_issue, embedding_vectors):
    """Combine multiple clustering approaches for better results"""
    
    print("\n🎯 Ensemble Clustering Approach")
    print("="*50)
    
    # Method 1: Embedding-based
    print("Running embedding-based clustering...")
    embedding_results = run_improved_clustering(performance_issue, embedding_vectors)
    
    # Method 2: Text-based
    print("\nRunning text-based clustering...")
    text_results = text_based_clustering_alternative(performance_issue)
    
    # Method 3: Hybrid approach - combine embeddings with TF-IDF
    print("\nRunning hybrid clustering...")
    
    # Normalize embeddings
    from sklearn.preprocessing import normalize
    norm_embeddings = normalize(embedding_vectors, norm='l2')
    
    # Get TF-IDF features
    texts = [str(text) for text in performance_issue]
    vectorizer = TfidfVectorizer(max_features=200, stop_words='english')
    tfidf_features = vectorizer.fit_transform(texts).toarray()
    
    # Combine features
    combined_features = np.hstack([norm_embeddings, tfidf_features])
    
    print(f"Combined features shape: {combined_features.shape}")
    
    # Cluster combined features
    analyzer = ImprovedClusteringAnalyzer(performance_issue)
    optimal_k, _ = analyzer.find_optimal_k_improved(combined_features)
    
    kmeans_hybrid = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
    hybrid_labels = kmeans_hybrid.fit_predict(combined_features)
    
    # Compare all three approaches
    print("\n📊 Ensemble Results Comparison:")
    print("="*50)
    
    methods = {
        'Embedding-based': embedding_results['labels'],
        'Text-based': text_results['labels'],
        'Hybrid': hybrid_labels
    }
    
    for name, labels in methods.items():
        if len(set(labels)) > 1:
            silhouette = silhouette_score(embedding_vectors, labels)
            print(f"{name}:")
            print(f"  Clusters: {len(set(labels))}")
            print(f"  Silhouette: {silhouette:.3f}")
            print()
    
    # Consensus clustering - use voting
    print("Creating consensus clustering...")
    
    # Convert all labels to same scale (0 to max_clusters-1)
    normalized_labels = {}
    for name, labels in methods.items():
        unique_labels = sorted(set(labels))
        label_map = {old: new for new, old in enumerate(unique_labels)}
        normalized_labels[name] = [label_map[label] for label in labels]
    
    # Simple voting mechanism
    consensus_labels = []
    for i in range(len(performance_issue)):
        votes = [normalized_labels[method][i] for method in methods.keys()]
        # Use most common vote, or first method if tie
        from collections import Counter
        vote_counts = Counter(votes)
        consensus_labels.append(vote_counts.most_common(1)[0][0])
    
    consensus_labels = np.array(consensus_labels)
    
    if len(set(consensus_labels)) > 1:
        consensus_silhouette = silhouette_score(embedding_vectors, consensus_labels)
        print(f"Consensus clustering:")
        print(f"  Clusters: {len(set(consensus_labels))}")
        print(f"  Silhouette: {consensus_silhouette:.3f}")
    
    # Final visualization
    analyzer.advanced_visualization(embedding_vectors, consensus_labels, performance_issue)
    
    return {
        'embedding_results': embedding_results,
        'text_results': text_results,
        'hybrid_labels': hybrid_labels,
        'consensus_labels': consensus_labels,
        'combined_features': combined_features
    }

# Performance optimization tips
def optimize_clustering_performance(embedding_vectors, performance_issue):
    """Additional performance optimization techniques"""
    
    print("\n⚡ Performance Optimization Tips")
    print("="*50)
    
    # 1. Sampling for large datasets
    if len(embedding_vectors) > 10000:
        print("Large dataset detected. Using sampling...")
        from sklearn.utils import resample
        
        sample_size = min(5000, len(embedding_vectors))
        sample_indices = resample(range(len(embedding_vectors)), 
                                n_samples=sample_size, 
                                random_state=42)
        
        sample_embeddings = embedding_vectors[sample_indices]
        sample_performance = performance_issue.iloc[sample_indices]
        
        print(f"Sampled {sample_size} items from {len(embedding_vectors)}")
        
        # Cluster on sample
        analyzer = ImprovedClusteringAnalyzer(sample_performance)
        processed_embeddings, _ = analyzer.preprocess_embeddings(sample_embeddings)
        optimal_k, _ = analyzer.find_optimal_k_improved(processed_embeddings)
        
        # Train on sample, predict on full dataset
        kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
        kmeans.fit(processed_embeddings)
        
        # Apply to full dataset
        full_processed, _ = analyzer.preprocess_embeddings(embedding_vectors)
        full_labels = kmeans.predict(full_processed)
        
        return full_labels, optimal_k
    
    # 2. Mini-batch K-means for very large datasets
    elif len(embedding_vectors) > 5000:
        print("Using Mini-batch K-means for better performance...")
        from sklearn.cluster import MiniBatchKMeans
        
        analyzer = ImprovedClusteringAnalyzer(performance_issue)
        processed_embeddings, _ = analyzer.preprocess_embeddings(embedding_vectors)
        optimal_k, _ = analyzer.find_optimal_k_improved(processed_embeddings[:1000])  # Sample for K selection
        
        mini_kmeans = MiniBatchKMeans(n_clusters=optimal_k, random_state=42, batch_size=1000)
        labels = mini_kmeans.fit_predict(processed_embeddings)
        
        return labels, optimal_k
    
    # 3. Hierarchical clustering with early stopping
    else:
        print("Using standard approach...")
        return run_improved_clustering(performance_issue, embedding_vectors)

# Main execution function
def main_clustering_pipeline(performance_issue, embedding_vectors):
    """Main pipeline that tries different approaches"""
    
    print("🎯 COMPREHENSIVE CLUSTERING PIPELINE")
    print("="*60)
    
    # Check data quality
    print(f"Data shape: {len(performance_issue)} items, {embedding_vectors.shape[1]} dimensions")
    print(f"Sample text: {str(performance_issue.iloc[0])[:100]}...")
    
    # Choose approach based on data size
    if len(embedding_vectors) > 10000:
        print("\n📊 Large dataset - using optimized approach")
        results = optimize_clustering_performance(embedding_vectors, performance_issue)
    elif len(embedding_vectors) > 1000:
        print("\n📊 Medium dataset - using ensemble approach")
        results = ensemble_clustering(performance_issue, embedding_vectors)
    else:
        print("\n📊 Small dataset - using comprehensive approach")
        results = run_improved_clustering(performance_issue, embedding_vectors)
    
    print("\n✅ Clustering pipeline completed!")
    return results


# main_clustering_pipeline(performance, embedding_vectors)


In [0]:
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import normalize, StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap

class DBSCANClustering:
    def __init__(self, performance_data):
        self.performance_data = performance_data
        self.embedding_vectors = None
        self.cluster_labels = None
        self.optimal_eps = None
        self.optimal_min_samples = None
        
    def preprocess_embeddings(self, embedding_vectors):
        """Preprocess embeddings for DBSCAN"""
        # Normalize embeddings (important for distance-based clustering)
        normalized_embeddings = normalize(embedding_vectors, norm='l2')
        
        # Optional: Standardize if embeddings have very different scales
        scaler = StandardScaler()
        standardized_embeddings = scaler.fit_transform(normalized_embeddings)
        
        print(f"Preprocessed embeddings shape: {standardized_embeddings.shape}")
        return standardized_embeddings, scaler
    
    def find_optimal_eps(self, embeddings, k=4):
        """Find optimal eps parameter using k-distance graph"""
        print(f"Finding optimal eps using {k}-distance graph...")
        
        # Calculate k-distances
        neighbors = NearestNeighbors(n_neighbors=k)
        neighbors_fit = neighbors.fit(embeddings)
        distances, indices = neighbors_fit.kneighbors(embeddings)
        
        # Sort distances
        distances = np.sort(distances, axis=0)
        distances = distances[:, k-1]  # k-th nearest neighbor distances
        
        # Plot k-distance graph
        plt.figure(figsize=(10, 6))
        plt.plot(distances)
        plt.title(f'{k}-Distance Graph')
        plt.xlabel('Data Points sorted by distance')
        plt.ylabel(f'{k}th Nearest Neighbor Distance')
        plt.grid(True)
        
        # Find elbow point (knee)
        def find_knee_point(distances):
            # Simple knee detection
            n_points = len(distances)
            all_coord = np.vstack((range(n_points), distances)).T
            first_point = all_coord[0]
            line_vec = all_coord[-1] - all_coord[0]
            line_vec_norm = line_vec / np.sqrt(np.sum(line_vec**2))
            
            vec_from_first = all_coord - first_point
            scalar_product = np.sum(vec_from_first * line_vec_norm, axis=1)
            vec_from_first_parallel = np.outer(scalar_product, line_vec_norm)
            vec_to_line = vec_from_first - vec_from_first_parallel
            dist_to_line = np.sqrt(np.sum(vec_to_line ** 2, axis=1))
            
            return np.argmax(dist_to_line)
        
        knee_idx = find_knee_point(distances)
        optimal_eps = distances[knee_idx]
        
        plt.axhline(y=optimal_eps, color='r', linestyle='--', 
                   label=f'Optimal eps = {optimal_eps:.3f}')
        plt.axvline(x=knee_idx, color='r', linestyle='--', alpha=0.5)
        plt.legend()
        plt.show()
        
        print(f"Suggested eps: {optimal_eps:.3f}")
        
        # Also suggest some alternative eps values
        percentiles = [80, 85, 90, 95]
        alternative_eps = [np.percentile(distances, p) for p in percentiles]
        
        print("Alternative eps values to try:")
        for p, eps in zip(percentiles, alternative_eps):
            print(f"  {p}th percentile: {eps:.3f}")
        
        return optimal_eps, alternative_eps
    
    def find_optimal_min_samples(self, embeddings, eps_values):
        """Find optimal min_samples parameter"""
        print("Finding optimal min_samples...")
        
        # Rule of thumb: min_samples = 2 * dimensions, but let's test different values
        n_features = embeddings.shape[1]
        suggested_min_samples = max(2, min(2 * n_features, len(embeddings) // 50))
        
        # Test different min_samples values
        min_samples_range = [2, 3, 4, 5, max(5, suggested_min_samples//2), 
                           suggested_min_samples, suggested_min_samples*2]
        min_samples_range = [ms for ms in min_samples_range if ms < len(embeddings)//10]
        
        print(f"Testing min_samples values: {min_samples_range}")
        print(f"Suggested min_samples based on dimensions: {suggested_min_samples}")
        
        return suggested_min_samples, min_samples_range
    
    def parameter_grid_search(self, embeddings):
        """Grid search for optimal DBSCAN parameters"""
        print("Performing parameter grid search...")
        
        # Get parameter ranges
        optimal_eps, alternative_eps = self.find_optimal_eps(embeddings)
        suggested_min_samples, min_samples_range = self.find_optimal_min_samples(embeddings, [optimal_eps])
        
        # Create parameter grid
        eps_values = [optimal_eps] + alternative_eps
        eps_values = sorted(list(set(eps_values)))  # Remove duplicates and sort
        
        results = []
        
        print(f"Testing {len(eps_values)} eps values × {len(min_samples_range)} min_samples values...")
        
        for eps in eps_values:
            for min_samples in min_samples_range:
                try:
                    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
                    labels = dbscan.fit_predict(embeddings)
                    
                    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
                    n_noise = list(labels).count(-1)
                    
                    # Calculate silhouette score (only for non-noise points)
                    if n_clusters > 1 and n_noise < len(labels):
                        non_noise_mask = labels != -1
                        if np.sum(non_noise_mask) > 1:
                            silhouette = silhouette_score(embeddings[non_noise_mask], 
                                                        labels[non_noise_mask])
                        else:
                            silhouette = -1
                    else:
                        silhouette = -1
                    
                    results.append({
                        'eps': eps,
                        'min_samples': min_samples,
                        'n_clusters': n_clusters,
                        'n_noise': n_noise,
                        'noise_ratio': n_noise / len(labels),
                        'silhouette': silhouette,
                        'labels': labels
                    })
                    
                except Exception as e:
                    print(f"Error with eps={eps}, min_samples={min_samples}: {e}")
        
        # Convert to DataFrame for easier analysis
        results_df = pd.DataFrame(results)
        
        # Filter out bad results
        good_results = results_df[
            (results_df['n_clusters'] >= 2) & 
            (results_df['n_clusters'] <= len(embeddings)//10) &
            (results_df['noise_ratio'] < 0.5) &
            (results_df['silhouette'] > 0)
        ].copy()
        
        if len(good_results) == 0:
            print("⚠️ No good parameter combinations found. Using best available...")
            good_results = results_df[results_df['n_clusters'] >= 2].copy()
        
        # Sort by silhouette score
        good_results = good_results.sort_values('silhouette', ascending=False)
        
        print(f"\nTop 5 parameter combinations:")
        print(good_results.head()[['eps', 'min_samples', 'n_clusters', 'n_noise', 'silhouette']].to_string())
        
        # Select best parameters
        best_params = good_results.iloc[0]
        
        print(f"\n🎯 Best parameters:")
        print(f"  eps: {best_params['eps']:.3f}")
        print(f"  min_samples: {best_params['min_samples']}")
        print(f"  n_clusters: {best_params['n_clusters']}")
        print(f"  noise_ratio: {best_params['noise_ratio']:.3f}")
        print(f"  silhouette: {best_params['silhouette']:.3f}")
        
        return best_params, results_df
    
    def apply_dbscan(self, embeddings, eps=None, min_samples=None):
        """Apply DBSCAN with given or optimal parameters"""
        
        if eps is None or min_samples is None:
            print("Finding optimal parameters...")
            best_params, _ = self.parameter_grid_search(embeddings)
            eps = best_params['eps']
            min_samples = best_params['min_samples']
        
        print(f"Applying DBSCAN with eps={eps:.3f}, min_samples={min_samples}")
        
        dbscan = DBSCAN(eps=eps, min_samples=min_samples, n_jobs=-1)
        labels = dbscan.fit_predict(embeddings)
        
        # Analyze results
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = list(labels).count(-1)
        
        print(f"✅ DBSCAN Results:")
        print(f"  Number of clusters: {n_clusters}")
        print(f"  Number of noise points: {n_noise}")
        print(f"  Noise ratio: {n_noise/len(labels):.3f}")
        
        if n_clusters > 1 and n_noise < len(labels):
            non_noise_mask = labels != -1
            if np.sum(non_noise_mask) > 1:
                silhouette = silhouette_score(embeddings[non_noise_mask], labels[non_noise_mask])
                print(f"  Silhouette score: {silhouette:.3f}")
        
        self.cluster_labels = labels
        self.optimal_eps = eps
        self.optimal_min_samples = min_samples
        
        return labels
    
    def visualize_clusters(self, embeddings, labels):
        """Create comprehensive visualizations"""
        
        # Prepare data
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        
        # 1. PCA visualization
        pca = PCA(n_components=2, random_state=42)
        embeddings_2d_pca = pca.fit_transform(embeddings)
        
        # 2. UMAP visualization (better for clusters)
        reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
        embeddings_2d_umap = reducer.fit_transform(embeddings)
        
        # 3. t-SNE visualization
        tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(embeddings)-1))
        embeddings_2d_tsne = tsne.fit_transform(embeddings)
        
        # Create subplots
        fig, axes = plt.subplots(1, 3, figsize=(20, 6))
        
        methods = [
            ('PCA', embeddings_2d_pca, f'PC1 ({pca.explained_variance_ratio_[0]:.1%})', 
             f'PC2 ({pca.explained_variance_ratio_[1]:.1%})'),
            ('UMAP', embeddings_2d_umap, 'UMAP 1', 'UMAP 2'),
            ('t-SNE', embeddings_2d_tsne, 't-SNE 1', 't-SNE 2')
        ]
        
        for i, (method, coords, xlabel, ylabel) in enumerate(methods):
            # Use different colors for noise points
            colors = labels.copy()
            unique_labels = set(labels)
            
            scatter = axes[i].scatter(coords[:, 0], coords[:, 1], c=colors, 
                                    cmap='tab10', alpha=0.7, s=50)
            
            # Highlight noise points
            if -1 in labels:
                noise_mask = labels == -1
                axes[i].scatter(coords[noise_mask, 0], coords[noise_mask, 1], 
                              c='black', marker='x', s=50, alpha=0.8, label='Noise')
            
            axes[i].set_title(f'{method} - {n_clusters} Clusters')
            axes[i].set_xlabel(xlabel)
            axes[i].set_ylabel(ylabel)
            
            if -1 in labels:
                axes[i].legend()
        
        plt.tight_layout()
        plt.show()
        
        # Interactive UMAP plot
        df_umap = pd.DataFrame({
            'x': embeddings_2d_umap[:, 0],
            'y': embeddings_2d_umap[:, 1],
            'cluster': labels,
            'text': self.performance_data.astype(str).tolist()
        })
        
        # Color noise points differently
        df_umap['cluster_str'] = df_umap['cluster'].apply(
            lambda x: 'Noise' if x == -1 else f'Cluster {x}'
        )
        
        fig_umap = px.scatter(df_umap, x='x', y='y', color='cluster_str',
                             hover_data=['text'],
                             title=f'Interactive DBSCAN Clustering - {n_clusters} Clusters',
                             color_discrete_sequence=px.colors.qualitative.Set1)
        fig_umap.show()
        
        return embeddings_2d_umap
    
    def analyze_clusters(self, embeddings, labels):
        """Detailed cluster analysis"""
        
        cluster_df = pd.DataFrame({
            'text': self.performance_data,
            'cluster': labels,
            'embedding': [emb.tolist() for emb in embeddings]
        })
        
        print("\n🔍 Detailed DBSCAN Cluster Analysis:")
        print("="*60)
        
        # Analyze each cluster
        unique_clusters = sorted([c for c in set(labels) if c != -1])
        
        cluster_stats = []
        
        for cluster_id in unique_clusters:
            cluster_data = cluster_df[cluster_df['cluster'] == cluster_id]
            cluster_size = len(cluster_data)
            
            print(f"\n📊 Cluster {cluster_id} ({cluster_size} items, {cluster_size/len(self.performance_data)*100:.1f}%)")
            print("-" * 50)
            
            # Sample texts from cluster
            sample_texts = cluster_data['text'].head(5).tolist()
            for j, text in enumerate(sample_texts, 1):
                print(f"{j}. {str(text)[:150]}...")
            
            if cluster_size > 5:
                print(f"... and {cluster_size - 5} more items")
            
            # Cluster statistics
            cluster_embeddings = np.array(cluster_data['embedding'].tolist())
            centroid = np.mean(cluster_embeddings, axis=0)
            
            # Intra-cluster distances
            distances = [np.linalg.norm(emb - centroid) for emb in cluster_embeddings]
            
            stats = {
                'cluster_id': cluster_id,
                'size': cluster_size,
                'percentage': cluster_size/len(self.performance_data)*100,
                'avg_distance_to_centroid': np.mean(distances),
                'std_distance_to_centroid': np.std(distances),
                'density': cluster_size / (np.std(distances) + 1e-6)  # Rough density measure
            }
            cluster_stats.append(stats)
            
            print(f"Average distance to centroid: {stats['avg_distance_to_centroid']:.3f}")
            print(f"Std distance to centroid: {stats['std_distance_to_centroid']:.3f}")
            print(f"Cluster density: {stats['density']:.3f}")
        
        # Analyze noise points
        noise_data = cluster_df[cluster_df['cluster'] == -1]
        if len(noise_data) > 0:
            print(f"\n🔍 Noise Points ({len(noise_data)} items, {len(noise_data)/len(self.performance_data)*100:.1f}%)")
            print("-" * 50)
            
            sample_noise = noise_data['text'].head(5).tolist()
            for j, text in enumerate(sample_noise, 1):
                print(f"{j}. {str(text)[:150]}...")
        
        # Overall statistics
        stats_df = pd.DataFrame(cluster_stats)
        
        if len(stats_df) > 0:
            print(f"\n📈 Overall DBSCAN Statistics:")
            print(f"Number of clusters: {len(unique_clusters)}")
            print(f"Number of noise points: {len(noise_data)}")
            print(f"Largest cluster: {stats_df['size'].max()} items ({stats_df['percentage'].max():.1f}%)")
            print(f"Smallest cluster: {stats_df['size'].min()} items ({stats_df['percentage'].min():.1f}%)")
            print(f"Average cluster size: {stats_df['size'].mean():.1f} items")
            print(f"Most dense cluster: {stats_df['density'].max():.3f}")
        
        return cluster_df, stats_df

# Usage function for DBSCAN
def run_dbscan_clustering(performance_issue, embedding_vectors):
    """Main function to run DBSCAN clustering"""
    
    print("🎯 DBSCAN CLUSTERING PIPELINE")
    print("="*50)
    
    # Initialize analyzer
    analyzer = DBSCANClustering(performance_issue)
    
    # Step 1: Preprocess embeddings
    print("\n1️⃣ Preprocessing embeddings...")
    processed_embeddings, scaler = analyzer.preprocess_embeddings(embedding_vectors)
    
    # Step 2: Apply DBSCAN
    print("\n2️⃣ Applying DBSCAN...")
    labels = analyzer.apply_dbscan(processed_embeddings)
    
    # Step 3: Visualize results
    print("\n3️⃣ Creating visualizations...")
    umap_coords = analyzer.visualize_clusters(processed_embeddings, labels)
    
    # Step 4: Analyze clusters
    print("\n4️⃣ Analyzing clusters...")
    cluster_df, stats_df = analyzer.analyze_clusters(processed_embeddings, labels)
    
    return {
        'analyzer': analyzer,
        'processed_embeddings': processed_embeddings,
        'labels': labels,
        'cluster_df': cluster_df,
        'stats_df': stats_df,
        'umap_coords': umap_coords
    }


In [0]:
performances

In [0]:
np.where(1)[0]

In [0]:
import numpy as np
from scipy.spatial.distance import cdist
import pandas as pd

def analyze_clusters_with_llm(embedding_vectors, cluster_labels, kmeans_model, original_texts, client):
    """
    For each cluster, select top 5 closest points to centroid and get LLM theme summary
    """
    cluster_themes = {}
    
    for cluster_id in range(kmeans_model.n_clusters):
        print(f"Analyzing Cluster {cluster_id}...")
        
        # Get points in this cluster
        # cluster_mask = cluster_labels == cluster_id
        cluster_indices = np.where(cluster_labels == cluster_id)[0]
        cluster_embeddings = embedding_vectors[cluster_indices]
        cluster_texts = [original_texts[i] for i in cluster_indices]
        
        # Calculate distances to centroid
        centroid = kmeans_model.cluster_centers_[cluster_id].reshape(1, -1)
        # distances = cdist(cluster_embeddings, centroid, metric='euclidean').flatten()
        distances = cdist(cluster_embeddings, centroid, metric='cosine').flatten()
    
        # Get top 15 closest points
        closest_indices = np.argsort(distances)[:15]
        top_15_agent = [cluster_texts[i] for i in closest_indices]
        top_15_distances  = [distances[i] for i in closest_indices]
        
        # Create LLM prompt
        prompt = f"""
         You are an expert Data Analyst specializing in customer service operations. You have been given a set of 15 representative agent performance evaluations from a much larger group of similar cases.
         Please sumamrize the key theme of the group of agent performance, ensure that point out the most common issue and provide a recommendation for improvement. Please be specific for example, training on communication skills.
            Agent Performance:
            {chr(10).join([f"{i+1}. {text}" for i, text in enumerate(top_15_agent)])}

            Provide a concise analysis in this format:
            Key Improvement Areas: [A short, concise title for the common improvement areas for those agent]
            Recommendation: [Recommend one high-impact strategic change (e.g., a new policy, a specific training module, or a process improvement)]

            """

        try:
            # Get LLM analysis
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[
                    {"role": "system", "content": "You are a customer service analyst.  You are going to review the agent performance. "},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.1,
                max_tokens=4*1024
            )
            
            llm_analysis = response.choices[0].message.content
            
        except Exception as e:
            print(f"Error with LLM analysis for cluster {cluster_id}: {e}")
            llm_analysis = f"Error: Could not analyze cluster {cluster_id}"
        
        # Store results
        cluster_themes[cluster_id] = {
            'cluster_size': len(cluster_texts),
            'top_15_agent': top_15_agent,
            'top_15_distances': top_15_distances,
            'llm_analysis': llm_analysis
        }
        
        print(f"✓ Cluster {cluster_id}: {len(cluster_texts)} issues analyzed")
    
    return cluster_themes

# Run the analysis
print("Starting cluster analysis with LLM theme summarization...")
cluster_themes = analyze_clusters_with_llm(
    embedding_vectors=embedding_vectors,
    cluster_labels=cluster_labels, 
    kmeans_model=kmeans,
    original_texts=performances,
    client=client
)

# Display results
print("\n" + "="*80)
print("CLUSTER THEME ANALYSIS")
print("="*80)

theme_summary = []

for cluster_id, data in cluster_themes.items():
    print(f"\n🔍 CLUSTER {cluster_id} ({data['cluster_size']} issues)")
    print("-" * 50)
    
    # Display LLM analysis
    print("LLM THEME ANALYSIS:")
    print(data['llm_analysis'])
    
    # print(f"\nTOP 5 REPRESENTATIVE ISSUES:")
    # for i, (text, distance) in enumerate(zip(data['top_15_agent'], data['top_15_distances'])):
    #     print(f"{i+1}. Distance: {distance:.4f}")
    #     print(f"   Issue: {text}")
    #     print()
    
    # Extract theme info for summary table
    analysis_lines = data['llm_analysis'].split('\n')
    # theme_title = "Unknown"
    
    for line in analysis_lines:
        if line.startswith('Key Improvement Areas:'):
            improve = line.replace('Key Improvement Areas:', '').strip()
        # elif line.startswith('Summary:'):
        #     summary = line.replace('Summary:', '').strip()
        elif line.startswith('Recommendation:'):
            recommendation = line.replace('Recommendation:', '').strip()
    
    theme_summary.append({
        'Cluster': cluster_id,
        'Key Improvement Areas:': improve,
        # 'Summary': summary,
        'Recommendation': recommendation,
        'Size': data['cluster_size'],
        'Percentage': f"{(data['cluster_size'] / len(performances) * 100):.1f}%",
        'Avg_Distance': f"{np.mean(data['top_15_distances']):.4f}"
    })

# Create summary table
summary_df = pd.DataFrame(theme_summary)
summary_df = summary_df.sort_values('Size', ascending=False)

print("\n" + "="*80)
print("CLUSTER THEMES SUMMARY")
print("="*80)
print(summary_df.to_string(index=False))

# Visualize the themes
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Cluster sizes with themes
bars = ax1.bar(summary_df['Cluster'], summary_df['Size'])
ax1.set_xlabel('Cluster ID')
ax1.set_ylabel('Number of Issues')
ax1.set_title('Issues per Cluster Theme')
ax1.set_xticks(summary_df['Cluster'])

# Add theme labels on bars
for i, (bar, theme) in enumerate(zip(bars, summary_df['Key Improvement Areas:'])):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + 0.5,
             f'{int(height)}', ha='center', va='bottom')
    # Add theme as x-axis label
    ax1.text(bar.get_x() + bar.get_width()/2., -max(summary_df['Size'])*0.1,
             theme[:15] + '...' if len(theme) > 15 else theme, 
             ha='center', va='top', rotation=45, fontsize=8)

# Priority distribution
# priority_counts = summary_df['Priority'].value_counts()
# colors = {'High': 'red', 'Medium': 'orange', 'Low': 'green'}
# pie_colors = [colors.get(p, 'gray') for p in priority_counts.index]

# ax2.pie(priority_counts.values, labels=priority_counts.index, autopct='%1.1f%%', 
#         colors=pie_colors)
# ax2.set_title('Priority Distribution')

plt.tight_layout()
plt.show()




In [0]:
summary_df